## How to use this notebook

Run notebook 01 first, then run this one top to bottom. It builds `data/processed/alias_map.json`
and adds `canonical_ingredients` to `data/processed/cleaned_recipes.parquet`. The maintained alias
definitions live in `src/alias_map.py`; compound-aware matching lives in `src/ingredient_utils.py`.

# 02 - Filipino ingredient alias map
Input: `data/processed/cleaned_recipes.parquet` (from notebook 01). Needs `ingredient_utils.py` in the same folder as this notebook.

Outputs: `data/processed/alias_map.json` and an updated `cleaned_recipes.parquet` with `canonical_ingredients` and `unmapped_terms` columns.

**Workflow:** run the seed map, look at what it misses in your data, add entries, rerun. Repeat 2-4 times.

In [2]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Working directory:", Path.cwd())

Working directory: c:\Users\micheala\Desktop\pinoy-recipe-rag\NLP-Mini-Project


## 1. Load

In [3]:
import json, re
import pandas as pd
from collections import Counter
from rapidfuzz import process, fuzz
from ingredient_utils import line_to_phrases, build_lookup, match_canonicals, normalize_text
import ingredient_utils
print(ingredient_utils.__file__)
print("cored" in ingredient_utils.PREP, "stemmed" in ingredient_utils.PREP)

clean = pd.read_parquet("data/processed/cleaned_recipes.parquet")
print(clean.shape)

c:\Users\micheala\Desktop\pinoy-recipe-rag\NLP-Mini-Project\src\ingredient_utils.py
True True
(1983, 16)


## 2. The alias map (edit this cell, then rerun everything below it)
Format: `canonical name: [aliases]`. Rules used here:
- Canonical is the common English name when one exists; Filipino-only items (kangkong, sotanghon) stay as they are.
- Only true synonyms and spelling variants go together. Cuts and varieties stay separate (pork belly is not merged into pork).
- Everything lowercase. Plurals are handled automatically, so skip them.
- Ambiguous words (e.g. `sili`, which can mean several chilies) are left out on purpose until you decide.

In [4]:
ALIASES = {
    # sauces, seasonings, pantry
    "soy sauce": ["toyo", "soya sauce", "dark soy sauce", "light soy sauce"],
    "fish sauce": ["patis"],
    "vinegar": ["suka", "cane vinegar", "white vinegar", "coconut vinegar"],
    "salt": ["asin"],
    "sugar": ["asukal", "white sugar", "granulated sugar"],
    "black pepper": ["paminta", "ground black pepper", "peppercorn", "black peppercorn", "whole peppercorn"],
    "cooking oil": ["mantika", "vegetable oil", "canola oil", "=oil"],
    "chili pepper": ["red chili pepper", "green chili pepper", "red chili", "green chili", "=chili"],    "shrimp paste": ["bagoong alamang", "bagoong"],
    "annatto": ["atsuete", "achuete", "achiote", "annatto seed", "achuete seed", "annatto powder"],
    "tamarind": ["sampalok"],
    "flour": ["harina", "all purpose flour"],
    "water": ["tubig"],
    "butter": ["mantikilya"],
    "cheese": ["keso"],
    "milk": ["gatas"],
    "coconut milk": ["gata"],
    "rice": ["bigas", "kanin", "steamed rice", "cooked rice"],
    "peanut": ["mani"],

    # aromatics and vegetables
    "garlic": ["bawang", "bawang na puti"],
    "onion": ["sibuyas", "red onion", "white onion", "yellow onion"],
    "spring onion": ["green onion", "scallion", "dahon ng sibuyas"],
    "ginger": ["luya"],
    "tomato": ["kamatis"],
    "string beans": ["sitaw", "snake bean", "long green bean", "yardlong bean", "yard long bean"],
    "water spinach": ["kangkong"],
    "eggplant": ["talong"],
    "bitter melon": ["ampalaya", "bitter gourd"],
    "squash": ["kalabasa", "pumpkin"],
    "cabbage": ["repolyo"],
    "radish": ["labanos", "daikon"],
    "taro": ["gabi"],
    "sweet potato": ["camote", "kamote"],
    "chayote": ["sayote"],
    "moringa": ["malunggay", "moringa leaves", "malunggay leaves"],
    "bottle gourd": ["upo"],
    "bok choy": ["pechay", "petsay", "pak choi"],
    "potato": ["patatas"],
    "carrot": ["karot"],
    "calamansi": ["kalamansi", "calamondin"],
    "lemongrass": ["tanglad"],
    "bay leaf": ["dahon ng laurel", "laurel leaf", "laurel leaves", "dried bay leaf"],
    "bird's eye chili": ["siling labuyo", "labuyo", "thai chili"],
    "long green pepper": ["siling haba", "siling pangsigang", "finger chili", "long chili", "long green chili"],
    "bean sprouts": ["togue", "taugeh"],
    "mung bean": ["monggo", "mongo"],
    "tofu": ["tokwa"],
    "banana blossom": ["puso ng saging"],
    "saba banana": ["saging na saba", "saba"],
    "jackfruit": ["langka", "nangka"],
    "pineapple": ["pinya"],
    "mango": ["mangga"],

    # meat, seafood, eggs
    "pork": ["baboy"],
    "chicken": ["manok"],
    "beef": ["baka"],
    "egg": ["itlog"],
    "pork blood": ["dugo ng baboy"],
    "liver": ["atay"],
    "milkfish": ["bangus", "bangus milkfish"],
    "shrimp": ["hipon", "prawn"],
    "squid": ["pusit"],
    "mussel": ["tahong"],
    "crab": ["alimango", "alimasag"],
    "anchovy": ["dilis"],
    "dried fish": ["tuyo"],

    # noodles
    "sotanghon": ["glass noodle", "cellophane noodle", "bean thread noodle"],
}

ALIASES["black pepper"] += ["=pepper"]          # exact match only: bare "pepper" means black pepper

ALIASES.update({
    # from your unmatched list
    "bell pepper": ["red bell pepper", "green bell pepper", "yellow bell pepper", "capsicum"],
    "chili pepper": ["red chili pepper", "green chili pepper", "red chili", "green chili", "chili"],
    "cornstarch": ["corn starch"],
    "coconut cream": ["kakang gata"],
    "oyster sauce": [],
    "okra": [],
    "cilantro": ["coriander leaves", "wansoy"],
    "raisin": ["pasas"],
    "banana ketchup": ["banana catsup", "banana sauce"],
    "star anise": [],
    "spinach": ["baby spinach"],
    "green beans": ["baguio beans", "french beans"],
    "mayonnaise": ["mayo"],
    "liquid seasoning": [],
    "paprika": [],
    "celery": [],
    "lime": [],
    "liver spread": [],
})

ALIASES.update({
    "rice flour": [], "rice noodles": [], "rice vinegar": [],
    "coconut water": [], "water chestnut": [],
    "sesame oil": [], "olive oil": [], "coconut oil": [],
    "evaporated milk": [], "condensed milk": [],
    "brown sugar": [], "powdered sugar": [],
})
ALIASES["bean sprouts"] += ["mung bean sprouts", "toge"] 

ALIASES["black pepper"] += ["ground pepper"]
ALIASES["string beans"] += ["long bean"]                 # long beans = sitaw
ALIASES["pork"] += ["liempo"]                            # pork belly, kept under pork like other cuts
ALIASES["lime"] += ["lime juice"]
ALIASES["calamansi"] += ["calamansi juice"]

ALIASES.update({
    "parsley": [],
    "lemon": ["lemon juice"],
    "tilapia": [],
    "tuna": [],
    "shallot": [],
    "leek": [],
    "snow peas": [],
    "green peas": [],
    "curry powder": [],
    "worcestershire sauce": [],
    "chicharon": ["chicharron"],
    "chorizo de bilbao": [],
    "lechon": [],
})

## 3. Build the lookup and check for collisions
A collision means the same word is listed under two different canonical names. The list must be empty.

In [5]:
lookup, collisions = build_lookup(ALIASES)
print(len(ALIASES), "canonical ingredients,", len(lookup), "terms in lookup")
print("collisions:", collisions if collisions else "none")

108 canonical ingredients, 265 terms in lookup
collisions: none


## 4. Apply to the data and measure coverage
Every raw ingredient line is cleaned into phrases, then matched against the map. Words that do not match anything are counted as *leftover*; the frequent ones are your to-do list.

In [6]:
line_hits = Counter(); n_lines = n_matched = 0
leftover_words = Counter(); unmatched_phrases = Counter()
tok_total = tok_covered = 0

for lines in clean["ingredient_lines"]:
    for line in lines:
        n_lines += 1
        any_hit = False
        for ph in line_to_phrases(line):
            canon, left = match_canonicals(ph, lookup)
            ntok = len(ph.split())
            tok_total += ntok; tok_covered += ntok - len(left)
            if canon:
                any_hit = True
            else:
                unmatched_phrases[ph] += 1
            leftover_words.update(left)
        n_matched += any_hit

print(f"lines with at least one match: {n_matched}/{n_lines} ({n_matched/n_lines:.1%})")
print(f"tokens covered by the map:     {tok_covered}/{tok_total} ({tok_covered/tok_total:.1%})")

lines with at least one match: 20395/22485 (90.7%)
tokens covered by the map:     32058/44160 (72.6%)


## 5. Your to-do list: what the map misses
**Left: whole phrases that matched nothing. Right: individual words that stayed uncovered.** Go through the top of both lists. For each term decide: is it an alias of something already in the map (add to its list), a new ingredient (add a new entry), or just noise like a cut or adjective (ignore).

In [7]:
print("TOP UNMATCHED PHRASES")
for ph, n in unmatched_phrases.most_common(60):
    print(f"{n:4d}  {ph}")
print("\nTOP LEFTOVER WORDS")
for w, n in leftover_words.most_common(60):
    print(f"{n:4d}  {w}")

TOP UNMATCHED PHRASES
  24  lengths
  21  parts
  18  gutted
  18  button mushrooms
  17  red pepper flakes
  16  cracked
  15  green olives
  15  green papaya
  15  leaves separated
  14  mustard leaves
  14  scaled
  13  basil leaves
  13  hot sauce
  13  honey
  13  kamias
  12  margarine
  12  bacon
  12  sinigang sa sampaloc mix
  12  cucumber
  12  pickle relish
  12  kesong puti
  12  lumpia wrappers
  12  serve
  11  salmon
  11  stems
  11  diagonally
  11  young corn
  11  on bias
  10  miki noodles
  10  turmeric powder
  10  knorr ginataang gulay mix
  10  in
  10  ground cumin
  10  lapu lapu
  10  all purpose cream
  10  corn oil
   9  chinese sausage
   9  vermicelli noodles
   9  shiitake mushrooms
   9  pimiento
   9  white pepper
   9  rings
   9  banana
   9  rounds
   9  coconut
   9  longganisa
   9  kinchay
   9  lumpia wrapper
   9  cream
   8  pepper corn
   8  bread crumbs
   8  pepper leaves
   8  cauliflower
   8  tinapa flakes
   8  mirin
   8  ham
   8  sin

## 6. Spelling-variant suggestions
For frequent leftover words, find the closest existing term. Suggestions only; accept them by editing the map in section 2. Check each one, because close spelling does not always mean the same ingredient.

In [8]:
known = list(lookup)
for w, n in leftover_words.most_common(80):
    best = process.extractOne(w, known, scorer=fuzz.ratio, score_cutoff=80)
    if best:
        print(f"{w!r:20} ({n}x)  ~  {best[0]!r} -> {lookup[best[0]]!r}  [{best[1]:.0f}]")

'pepper'             (183x)  ~  '=pepper' -> 'black pepper'  [92]
'peppers'            (83x)  ~  '=pepper' -> 'black pepper'  [86]
'oil'                (83x)  ~  '=oil' -> 'cooking oil'  [86]
'salted'             (39x)  ~  'salt' -> 'salt'  [80]
'parts'              (33x)  ~  'patis' -> 'fish sauce'  [80]
'sampaloc'           (30x)  ~  'sampalok' -> 'tamarind'  [88]
'baby'               (29x)  ~  'baboy' -> 'pork'  [89]


## 7. Sanity tests on known cases

In [9]:
tests = {
    "toyo": "soy sauce", "patis": "fish sauce", "bawang": "garlic",
    "sitaw": "string beans", "snake beans": "string beans", "long green beans": "string beans",
    "Sibuyas": "onion", "4 cloves garlic, peeled and chopped": "garlic",
    "dahon ng laurel": "bay leaf", "kalamansi": "calamansi",
}
for text, want in tests.items():
    got, _ = normalize_text(text, lookup)
    print("OK  " if want in got else "FAIL", f"{text!r:42} -> {got}")

for q in ["toyo, bawang, sibuyas, manok", "baboy, sitaw, kamatis, patis"]:
    print("\nquery:", q)
    got = []
    for item in q.split(","):
        got += normalize_text(item, lookup)[0]
    print("  ->", got)

OK   'toyo'                                     -> ['soy sauce']
OK   'patis'                                    -> ['fish sauce']
OK   'bawang'                                   -> ['garlic']
OK   'sitaw'                                    -> ['string beans']
OK   'snake beans'                              -> ['string beans']
OK   'long green beans'                         -> ['string beans']
OK   'Sibuyas'                                  -> ['onion']
OK   '4 cloves garlic, peeled and chopped'      -> ['garlic']
OK   'dahon ng laurel'                          -> ['bay leaf']
OK   'kalamansi'                                -> ['calamansi']

query: toyo, bawang, sibuyas, manok
  -> ['soy sauce', 'garlic', 'onion', 'chicken']

query: baboy, sitaw, kamatis, patis
  -> ['pork', 'string beans', 'tomato', 'fish sauce']


## 8. Save `alias_map.json`, and add the canonical columns to the recipes
Only run this when the coverage and to-do lists look good enough. It is fine to save an early version and come back later: just rerun sections 2 to 8.

In [10]:
with open("data/processed/alias_map.json", "w", encoding="utf-8") as f:
    json.dump({k: sorted(set(v)) for k, v in sorted(ALIASES.items())}, f, ensure_ascii=False, indent=2)

def row_canonicals(row):
    canon, left = [], []
    for text in list(row["ingredient_lines"]) + list(row["ingredient_list"]):
        c, l = normalize_text(text, lookup)
        canon += [x for x in c if x not in canon]
        left += l
    return canon, sorted(set(left))

res = clean.apply(row_canonicals, axis=1)
clean["canonical_ingredients"] = [r[0] for r in res]
clean["unmapped_terms"] = [r[1] for r in res]

print("recipes with zero canonical ingredients:", (clean["canonical_ingredients"].str.len() == 0).sum())
print("average canonical ingredients per recipe:", clean["canonical_ingredients"].str.len().mean().round(1))
clean.to_parquet("data/processed/cleaned_recipes.parquet", index=False)
clean[["recipe_name", "canonical_ingredients"]].sample(min(5, len(clean)), random_state=1)

recipes with zero canonical ingredients: 2
average canonical ingredients per recipe: 10.0


,recipe_name,canonical_ingredients
820,Adobong Sitaw with Pork,"[cooking oil, pork, onion, garlic, vinegar, so..."
1227,Pork Nilaga,"[onion, pork, water, fish sauce, black pepper,..."
1419,Lemongrass Pork Chop,"[garlic, onion, lemongrass, black pepper, salt..."
111,Pork Ginataan,"[pork, spinach, coconut milk, coconut cream, b..."
480,Lumpiang Sariwa (Fresh Spring Rolls Recipe),"[sweet potato, fish sauce, cabbage, carrot, wa..."
